In [1]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print('Model is ready to use!')

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model is ready to use!


In [4]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="Please give me list of only names of top 5 countries by their economy size"
)
prompt = prompt_template.format()

response = model.invoke(prompt)
print(response)

content='United States  \nChina  \nJapan  \nGermany  \nIndia' additional_kwargs={} response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-10-26T09:17:21.572187373Z', 'done': True, 'done_reason': 'stop', 'total_duration': 678341014, 'load_duration': None, 'prompt_eval_count': 83, 'prompt_eval_duration': None, 'eval_count': 134, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'} id='lc_run--01a17780-0229-7eb3-91d1-160d1732c1bf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 83, 'output_tokens': 134, 'total_tokens': 217}


In [ ]:
from langchain_core.output_parsers import StrOutputParser

string_output_parser = StrOutputParser()

prompt = PromptTemplate(
    template="Write a poem on the city of Paris"
)

chain = prompt | model | string_output_parser
response = chain.invoke({})
print(type(response))
print(response)


<class 'langchain_core.messages.ai.AIMessage'>
content='**Paris, in the Whisper of Light**\n\nBeneath the vaulted sigh of morning,\nthe Seine slides silver‑threaded\nthrough cobblestones that have heard\nthe clatter of carriage wheels\nand the hushed tread of poets’ shoes.\n\nThe city rises, a rose‑petaled echo,\nits balconies breathing perfume\nof fresh baguette and newly‑bloomed lilacs.\nEvery window frames a painted sky,\nevery streetlamp a lantern‑kissed promise.\n\nAt the heart, the Eiffel bows,\nsteel ribs stretching toward the clouds,\na lattice of dreaming iron that\ngathers starlight for the night\nwhen lovers trace the river’s curve\nwith fingertips and whispered vows.\n\nMontmartre, the wind‑swept cap,\nholds the ghosts of painters,\ntheir canvases still wet with\nthe lavender of twilight,\nwhile the cafés spill laughter\ninto the rain‑slicked alleys.\n\nIn the cathedral’s hush,\nthe rose‑gold rose windows bleed\ncolor onto stone—amber, indigo,\na hymn of fire that climbs\nt

In [9]:
from langchain_core.output_parsers import ListOutputParser
from langchain_core.prompts import PromptTemplate

class CustomListOutputParser(ListOutputParser):
    def parse(self, text: str) -> list:
        # Custom parsing logic to extract list items from the text
        items = [item.strip() for item in text.split('\n') if item.strip()]
        return items

prompt = PromptTemplate(
    template="Please give me list of only names of top 5 countries by their economy size"
)
list_output_parser = CustomListOutputParser()
chain = prompt | model | list_output_parser
response = chain.invoke({})
print(response)

['United States', 'China', 'Japan', 'Germany', 'India']


In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

prompt = PromptTemplate(
    template="Extract the person's details from the following text: \nJohn Doe, 30 years old, lives in New York."
)

# Output schema for the extracted details
class Person(BaseModel):
    name: str = Field("Person's name")
    age: int = Field("Person's age")
    city: str = Field("Person's city")
json_output_parser = JsonOutputParser(pydantic_object=Person)
chain = prompt | model | json_output_parser
response = chain.invoke({})
print(response)

{'name': 'John Doe', 'age': 30, 'city': 'New York'}


In [22]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

prompt = PromptTemplate(
    template="Extract the person's details from the following text: \nJohn Doe, 30 years old, lives in New York."
)

# Output schema for the extracted details
class Person(BaseModel):
    name: str = Field("Person's name")
    age: int = Field("Person's age")
    city: str = Field("Person's city")
pydantic_output_parser = PydanticOutputParser(pydantic_object=Person)
chain = prompt | model | pydantic_output_parser
response = chain.invoke({})
print(type(response))
print(response)

<class '__main__.Person'>
name='John Doe' age=30 city='New York'


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

pydantic_output_parser = PydanticOutputParser(pydantic_object=Person)

prompt = PromptTemplate(
    template="Extract the person's details from the following text and " \
    "convert it into the format instructions {format_instructions}: \nJohn Doe, {age} years old, lives in New York.",
    input_variables=["age"],
    partial_variables={
        "format_instructions": pydantic_output_parser.get_format_instructions()
    }
)

# Output schema for the extracted details
class Person(BaseModel):
    name: str = Field("Person's name")
    age: int = Field("Person's age")
    city: str = Field("Person's city")
chain = prompt | model | pydantic_output_parser
response = chain.invoke({"age": 30})
print(type(response))
print(response)

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.